# Aashish AI — Stage A training (§7.3, P4)

Drives the `training/scripts/` modules, which are tested without a GPU. This notebook is the thin part: it sequences them, checks the environment, and **stops on the licence gate**.

**Status: never executed end to end.** Everything it calls is covered by `npm run test:all`, and reading it against the modules found two defects that running it would have taken hours to reveal — both fixed (the notes are on the cells below). But no Kaggle session has run yet, so treat the first run as the test. Two gates stay UNVERIFIED until it does: *the loss decreases* and *training resumes*.

**Read this before the first run:** the corpus is chosen by the licence gate in §2, and two of the four obvious candidates are NonCommercial. See `docs/DATA_LICENSES.md`.

## 1. Setup

In [ ]:
!nvidia-smi

import subprocess
import sys

import torch

print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('python', sys.version.split()[0], '| commit',
      subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                     capture_output=True, text=True).stdout.strip())
print('bf16 supported:',
      torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
# A T4 has no bf16 — that is exactly why §7.5 specifies fp16 AMP.

In [ ]:
import os
import pathlib
import shutil

WORK = pathlib.Path('/kaggle/working')
MOUNT = pathlib.Path('/kaggle/input')
HOME = WORK / 'aashish-ai-portfolio'


def find_repo():
    """Where this session put the repository.

    A hard-coded list of guesses was wrong twice over. It preferred the
    read-only Dataset mount, and it assumed a Dataset appears at
    `/kaggle/input/<name>` — while Kaggle mounts "Your Datasets" under
    `/kaggle/input/datasets/<username>/<slug>/`, so `training/` was never where
    the first guess looked. That is the whole reason this cell now discovers the
    tree instead of listing places it might be.

    The direct paths stay, because they are instant and they cover a clone. The
    fallback is a file the repository is known to contain, which is what makes
    this survive the next time the layout changes.
    """
    for candidate in (HOME, WORK / 'newportfolio'):
        if (candidate / 'training').is_dir():
            return candidate
    for marker in MOUNT.rglob('package.json'):
        if (marker.parent / 'training' / 'notebooks').is_dir():
            return marker.parent
    return None


repo = find_repo()
if repo is None:
    raise SystemExit(
        f'No repository found. Looked in {HOME} and {WORK / "newportfolio"}, and '
        f'for any package.json under {MOUNT}. Attach the aashish-ai-portfolio '
        f'dataset, or clone the branch into {HOME}.')

# A Dataset mount is read-only, and this run writes data/sources.json (the
# licence verification itself), data/raw/, data/extracted/, data/processed/ and
# the checkpoints long before it trains anything: from the mount, the first
# write dies with OSError: [Errno 30] and the GPU is already allocated. So copy
# it out first, then prove the copy is writable rather than assuming it.
os.chdir(repo)
if pathlib.Path.cwd().is_relative_to(MOUNT):
    if not HOME.exists():
        shutil.copytree(pathlib.Path.cwd(), HOME)
    os.chdir(HOME)
pathlib.Path('.tmp-writable').write_text('ok')
pathlib.Path('.tmp-writable').unlink()

print('cwd:', os.getcwd())
# `python -m` from here puts the repo root on sys.path, which is why every
# command below is run as a module rather than as a path to a file.
!python -c "import tokenizers, numpy, pyarrow; print('tokenizers', tokenizers.__version__, '| numpy', numpy.__version__, '| pyarrow', pyarrow.__version__)"
!git rev-parse --short HEAD


## 2. Licence gate (§7.3)

A source cannot be downloaded until a named person records the licence they read. There is no `--force`, and this is the **first wall, not a formality**: on the first pass it found L3Cube-HingCorpus and DailyDialog are both NonCommercial.

In [ ]:
!python -m training.scripts.fetch_corpus --check

# The output prints the exact --verify command for each source, including the
# licence that was already looked up and the page it was found on. Verifying is
# a signature, so it belongs to the owner:
#
# !python -m training.scripts.fetch_corpus --verify hindi_wikipedia --spdx CC-BY-SA-4.0 --url https://dumps.wikimedia.org/legal.html --reviewer "Aashish Kumar"
#
# A NonCommercial licence cannot be verified into usability: --verify records the
# fact and leaves the source disabled.

In [ ]:
# Only sources the gate reports as `ready` can be fetched. Downloads are
# resumable (.part + Range) and hashed, so a restarted session continues.
SOURCES = ['hindi_wikipedia']  # add the others once their licences are verified

for source in SOURCES:
    !python -m training.scripts.fetch_corpus --source {source}

!ls -la data/raw/*/fetch_manifest.json

## 3. Extraction — a download is not text

**This step did not exist, and its absence is why the pipeline could not run.** `fetch_corpus` writes `hiwiki-latest-pages-articles.xml.bz2`; `prepare_data` globs for `*.txt`. Both were tested; nothing connected them.

Text goes to `data/extracted/`, one document per line. That directory is shared on purpose: `prepare_data --raw` reads a *single* directory, so several sources meeting there is how a multi-source corpus is assembled.

In [ ]:
!python -m ai.data.extract --check

for source in SOURCES:
    !python -m ai.data.extract --source {source} --min-chars 200

# For a parquet corpus (TinyStories, Sangraha), inspect the columns rather than
# guessing the text column:
# !python -m ai.data.extract --in data/raw/tinystories --list-columns
# !python -m ai.data.extract --source tinystories --kind parquet --text-column text --min-chars 200
#
# --min-chars is corpus-dependent. 200 suits articles; a short-story or dialogue
# corpus needs less, and extracting 0 documents is a named failure rather than
# an empty file precisely so that this is obvious.

## 4. Corpus pipeline (§7.3) — pass 1: read the stats

Without `--tokenizer`, this pass writes cleanup statistics, the leakage report and `corpus.txt` but **no shards**. It is the cheap pass, and its output is the one worth reading before spending GPU hours.

In [ ]:
!python -m training.scripts.prepare_data --raw data/extracted --out data/processed/stage_a

!cat data/processed/stage_a/stats.json
!cat data/processed/stage_a/leakage.json

# Read `dropped` by reason. A corpus that keeps 4% of its lines is a filtering
# problem, not a training problem, and no hyperparameter will fix it.
# `corpus.txt` is the train split of the *processed* text — what the tokenizer
# learns from next.

## 5. Tokenizer (§7.2) — train, then freeze

In [ ]:
TOKENIZER = 'ai/tokenizer/artifacts/stage-a-12k'

!python -m ai.tokenizer.train --corpus "data/processed/stage_a/corpus.txt" --vocab-size 12288 --min-frequency 2 --out $TOKENIZER

# Trained on corpus.txt — the train split of the processed text — and not on the
# raw download. Two reasons: a vocabulary learned from text the filters are
# about to reject describes a corpus that never existed, and one that saw val
# text lets the validation set shape the thing it is scored against.
#
# The trainer refuses a vocabulary the corpus cannot support instead of quietly
# emitting a smaller one: the ceiling is distinct word forms, not bytes. If this
# fails, the corpus needs more lexical variety, not a smaller vocab.

In [ ]:
!python -m ai.tokenizer.fertility $TOKENIZER

import json

meta = json.load(open(f'{TOKENIZER}/meta.json'))
print('frozen tokenizer_version:', meta['tokenizer_version'], '| vocab', meta['vocab_size'])
print('placeholders reserved:', meta['placeholder_count'])

# tokens/word is the number to judge here: a high Hindi figure means the
# vocabulary is spending its budget on English, and every Hindi document then
# costs more of the block size than an English one.

## 6. Corpus pipeline — pass 2: shard with the frozen tokenizer

**Sharding depends on the tokenizer, so it cannot run before §5.** This notebook originally sharded with `seed-1k` and then trained with the 12k tokenizer. Every token id from a 1k vocabulary is still a legal index into a 12k embedding, so nothing failed — the run would have produced a model that encodes text one way and was trained another, and no downstream check could have detected it.

`train_smoke` now compares the shard manifest's `vocab_size` against the tokenizer's and refuses a mismatch. Two passes are safe because the split is a content hash, not a shuffle.

In [ ]:
!python -m training.scripts.prepare_data --raw data/extracted --out data/processed/stage_a --tokenizer $TOKENIZER

!cat data/processed/stage_a/shards/manifest.json

# Check that `tokenizer_version` here matches the value printed in §5. It is the
# field the guard reads, and it is there so that this question can be answered
# by looking rather than by remembering.

## 7. Model + parameter count (§7.1)

In [ ]:
!python inference/count_parameters.py --config A --gate

# Two derivations of one number: analytic from the config, then materialised
# from the real module's state_dict, name by name and shape by shape. A mismatch
# exits non-zero — which is the only reason to trust the analytic count.

## 8. Smoke test first (§7.5)

Cheap, and it fails fast on a corpus or tokenizer problem that would otherwise surface 40 minutes into a real run.

In [ ]:
RUN = '/kaggle/working/checkpoints/smoke'

!python -m training.scripts.train_smoke --steps 50 --save-every 10 --amp --shards data/processed/stage_a/shards --tokenizer $TOKENIZER --run-dir $RUN --json /kaggle/working/smoke.json --gate

# ...then prove the resume, in the same session. It must continue the recorded
# stream rather than restart it, and the manifest must report the tokens already
# consumed.
!python -m training.scripts.train_smoke --steps 80 --save-every 10 --amp --shards data/processed/stage_a/shards --tokenizer $TOKENIZER --run-dir $RUN --resume auto --gate

# `--json` also records tokens_per_second. That measurement is what makes §9 a
# decision instead of a guess, so do not skip the smoke run to save ten minutes.

## 9. Budget decision (§7.3)

§7.3's ~20 tokens/param is a reference, not a requirement. This turns one measurement into the number of hours the run actually costs.

In [ ]:
!python -m training.scripts.estimate_budget --config A --from-run /kaggle/working/smoke.json --hours 9 --dataset-tokens data/processed/stage_a/shards

# Kaggle's free tier gives roughly 30 GPU-hours a week, so size the run to the
# quota. A T4 measures far slower than a desktop GPU: use the printed number,
# not an intuition from a local card.

## 10. Stage A training, resumable

In [ ]:
# Persist checkpoints OUTSIDE the ephemeral session before training (§7.5):
#   · push /kaggle/working/checkpoints/stage-a to a Kaggle Dataset / Drive / HF Hub, or
#   · write to /kaggle/working and Save Version every session
# A session that ends without either loses everything in this directory.
!python -m training.scripts.train_stage_a --config A --tokenizer $TOKENIZER --shards data/processed/stage_a/shards --run-dir /kaggle/working/checkpoints/stage-a --steps 20000 --batch 16 --block 1024 --grad-accum 4 --amp --gate --max-minutes 540 --resume auto

# --max-minutes stops cleanly and writes a checkpoint before the time box ends,
# so an interrupted run is a resume rather than a loss. 16 x 1024 x 4 = 65K
# tokens/step.

In [ ]:
# Every subsequent session: the identical command. It resumes from latest.pt and
# reports the step, the length of the loss history and the tokens consumed.
!ls -la /kaggle/working/checkpoints/stage-a
!cat /kaggle/working/checkpoints/stage-a/RUN_MANIFEST.json

# A resume that silently began a new run would look like success here, so check
# that the step continues from where the previous session stopped.

## 11. Validation and samples

In [ ]:
print('<|user|> What projects has Aashish built? <|asst|>')

# Deliberately not implemented yet. Stage A gives a base model that continues
# text; prompts in the §7.4 format are what P5's Stage B teaches it to answer,
# and sampling from the base model with turn tokens it has never learnt is a
# way to conclude the architecture is broken when the data is what is missing.
#
# The two numbers that belong here are the validation loss on the val shards and
# the loss curve. Both come from the training JSON — and a model that has learnt
# synthetic stories and simplewiki will read as fluent and slightly
# encyclopedic. That is a corpus property, not a model defect.

## 12. Export (P6)

In [ ]:
# P6: checkpoint to fp16, then inference/benchmark/benchmark_cpu.py, then the
# parity check, then GGUF/ONNX, then quantize, then measure.
#
# Not scaffolded here on purpose: it needs a checkpoint, and a pipeline built
# against a checkpoint that does not exist yet is a pipeline whose interface is
# invented. Sizes are measured, never estimated in prose.
print('P6 begins once a Stage A checkpoint exists.')